# Ouroboros training (Colab T4)

Runtime -> Change runtime type -> **T4 GPU**, then run all cells top to bottom.

Same notebook for the first session and after a disconnect: the data cell reuses the
token cache on Drive, and the train cell resumes from `latest.pt` if one exists.

Logs append to `MyDrive/ouroboros/main.log`. W&B is optional (see its cell).

In [ ]:
# check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# mount Drive: checkpoints and the token cache live here so they survive disconnects
from google.colab import drive

drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/ouroboros"
CKPT_DIR = f"{DRIVE}/checkpoints"
RUN_NAME = "main"
!mkdir -p {CKPT_DIR}

In [ ]:
# get the code (pull if this session already cloned it)
import os

if os.path.exists("/content/Ouroboros"):
    !git -C /content/Ouroboros pull
else:
    !git clone https://github.com/asherk7/Ouroboros.git /content/Ouroboros
%cd /content/Ouroboros
!pip install -q -e .

In [ ]:
# tokenizer + token cache: copy from Drive if saved, else build once (~10-20 min) and save.
# training reads it from local disk; shuffled reads through the Drive mount are slow
if os.path.exists(f"{DRIVE}/data/train.bin"):
    !cp -r {DRIVE}/data .
else:
    !python -c "import sys; sys.path.insert(0, 'training'); import train as t; c = t.TrainConfig(); t.train_tokenizer(c, t.OuroborosConfig().vocab_size); t.tokenize_split(c, 'train'); t.tokenize_split(c, 'validation')"
    !cp -r data {DRIVE}/
!ls -lh data

In [ ]:
# optional W&B: set True and add WANDB_API_KEY under Colab's 🔑 Secrets.
# without it, metrics go to stdout and to train.log on Drive (survives disconnects)
USE_WANDB = False

wandb_flag = ""
if USE_WANDB:
    from google.colab import userdata

    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    wandb_flag = f"--wandb-project ouroboros --run-name {RUN_NAME}"


In [ ]:
# train, resuming if a checkpoint exists. -u + tee: logs stream live and append to Drive
latest = f"{CKPT_DIR}/latest.pt"
resume = f"--resume {latest}" if os.path.exists(latest) else ""
print("resuming from", latest if resume else "nothing (fresh run)")

!python -u training/train.py --ckpt-dir {CKPT_DIR} {wandb_flag} {resume} 2>&1 | tee -a {DRIVE}/{RUN_NAME}.log

**Watch for the first ~15 min:** loss starts near 9.0 and falls; grad_norm mostly ≤ ~1;
rho_A < 1; n_loops varies; expert_load_var trends down; val_ppl every 500 steps.

ETA = (5000 - step) × 32768 / tokens_per_s seconds.